# Phase 3: RAG (retrieval-augmented generation)

A model only knows what it was trained on, so it guesses about documents it has never seen. RAG stores our documents as searchable chunks, retrieves the closest ones to a question, and gives them to the model.

Search works by meaning (embeddings), not keywords. 

Note: _The policy documents are fictional sample content._

In [ ]:
from langchain.agents import create_agent

from community_voice.llm.providers import get_chat_model
from community_voice.rag.store import build_index, search
from community_voice.tools.policy import search_policy

## 1. Build the index

Each `##` section becomes a chunk. Chroma embeds every chunk on your machine. The first run downloads the embedding model, so it can take a while.

In [ ]:
print(build_index(), "chunks indexed")

## 2. Search by meaning

The queries share few words with the documents. A smaller distance means a closer match.

In [ ]:
queries = [
    "No lights on our street at night",
    "How long until a burst pipe is fixed?",
    "My rubbish was not collected",
]
for query in queries:
    print(query)
    for hit in search(query, n_results=2):
        title = hit["text"].splitlines()[0]
        print(f"  {hit['distance']:.2f}  {title}")

## 3. Ask without retrieval

The model has never seen our charters, so it guesses.

In [ ]:
question = (
    "Under our service charter, within how many working days must a faulty "
    "streetlight at a dangerous intersection be repaired?"
)
print(get_chat_model("groq").invoke(question).text)

## 4. Ask with retrieval

Same question, but the agent can call `search_policy` and must answer from what it finds.

In [ ]:
agent = create_agent(
    model=get_chat_model("groq"),
    tools=[search_policy],
    system_prompt=(
        "Answer only from the search_policy results and name the source file. "
        "If the answer is not there, say so."
    ),
)
result = agent.invoke({"messages": [{"role": "user", "content": question}]})
print(result["messages"][-1].text)

## What to notice

- Search found the right section without matching keywords.
- Without retrieval the model guessed. With it, the answer comes from our document and names the source.
- Change a time in `data/knowledge_base/policies/water_charter.md`, run section 1 again, and ask again.